# Theme 3B: receipt extraction with fine-tuned Gemma 4 (Hacktoberfest Hack Day)

**Pipeline:** plain baseline, prompted baseline, LoRA fine-tune (Unsloth, vision), fine-tuned evaluation on the same 100 test receipts, error analysis, local-receipt robustness check, Gradio expense app.

**Runtime:** Runtime > Change runtime type > **G4** or **A100** (a T4 also works, more slowly). Run the sections **in order**. Each section ends with a **GATE**: check the printed values before moving on.

**Built in from the research:**
- Image before text in both training and inference (Gemma 4 model card).
- 560-token visual budget, verified by counting image tokens.
- Greedy decoding, one receipt at a time.
- Adapters saved separately, never merged.
- `pip freeze` straight after a successful install.

**Status:** written without GPU access, so it is untested. Anything that fails on the real runtime should be fixed in place, with the change noted in the training-settings table.

## 0. Install (run once per runtime, 3-5 min)

In [ ]:
%%capture install_log
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
!pip install --no-deps --upgrade timm
!pip install "gradio>=6,<7" pandas python-dateutil

In [ ]:
print(install_log.stdout[-1500:])   # tail of the install output; look for ERROR lines
import torch, transformers
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE", "| torch", torch.__version__, "| transformers", transformers.__version__)
!pip freeze > versions.txt
print("Saved versions.txt. If a later runtime breaks, reinstall from it with: pip install -r versions.txt")

## 1. Configuration (every setting you will report)

In [ ]:
MODEL_NAME = "unsloth/gemma-4-E2B-it"   # size comparison: "unsloth/gemma-4-E4B-it" in a second runtime
LOAD_IN_4BIT = False      # official Unsloth vision recipe: 16-bit LoRA. Set True only if memory is short (T4).
MAX_SOFT_TOKENS = 560     # visual token budget: 70 / 140 / 280 / 560 / 1120 (default 280 shrinks receipts to ~0.67x)
LORA_R, LORA_ALPHA = 32, 32
SMOKE_STEPS = 5
TRAIN_STEPS = 157         # one epoch: 626 receipts / (batch 1 x grad-accum 4)
LEARNING_RATE = 2e-4
SAVE_STEPS = 40           # checkpoints at 40/80/120 allow a free learning curve
N_TEST = 100              # fixed, seeded test subset of the 361 test receipts
SEED = 0
MAX_NEW_TOKENS = 256
FIELDS = ["company", "date", "address", "total"]
OUT = "/content/hackday"
INSTRUCTION = "Extract the company, date, address and total from this receipt. Reply with JSON only."
PROMPT_DETAILED = (
    "Extract four fields from this receipt. Reply with one JSON object only, with exactly these keys: "
    "company, date, address, total.\n"
    "Rules:\n"
    "- company: the business name exactly as printed, in capital letters.\n"
    "- date: the transaction date exactly as printed (keep its format, for example 25/12/2018 or 12-01-19).\n"
    "- address: the full business address exactly as printed, lines joined with a single space, in capital letters.\n"
    "- total: the final amount paid, digits with two decimals (for example 9.00), no currency symbol.\n"
    "No explanation, no markdown."
)
import os; os.makedirs(OUT, exist_ok=True)

## 2. Load the plain model

In [ ]:
from unsloth import FastVisionModel
import torch
model, processor = FastVisionModel.from_pretrained(MODEL_NAME, load_in_4bit=LOAD_IN_4BIT, use_gradient_checkpointing="unsloth")
try:
    from unsloth.chat_templates import get_chat_template
    processor = get_chat_template(processor, "gemma-4")      # as in the official Unsloth Gemma 4 vision notebook
except Exception as e:
    print("chat-template step skipped:", repr(e)[:200])
ip = getattr(processor, "image_processor", None)
print("image processor:", type(ip).__name__, "| token-related attributes:", [k for k in dir(ip) if "token" in k.lower() and not k.startswith("_")])
if ip is not None and hasattr(ip, "max_soft_tokens"):
    ip.max_soft_tokens = MAX_SOFT_TOKENS                     # UNVERIFIED route for the training collator; checked in section 8
    print("image_processor.max_soft_tokens set to", ip.max_soft_tokens)
IMAGE_TOKEN_ID = getattr(processor, "image_token_id", None)
print("image_token_id:", IMAGE_TOKEN_ID, "| GPU memory used: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

## 3. Data: SROIE (626 train / 361 test) and a fixed 100-receipt test subset

In [ ]:
import json, random
from datasets import load_dataset
train_data = load_dataset("jsdnrs/ICDAR2019-SROIE", split="train")
test_data = load_dataset("jsdnrs/ICDAR2019-SROIE", split="test")
print(len(train_data), "train /", len(test_data), "test receipts")
test_idx = sorted(random.Random(SEED).sample(range(len(test_data)), N_TEST))
test_subset = [test_data[i] for i in test_idx]
json.dump([s["key"] for s in test_subset], open(f"{OUT}/test_keys.json", "w"))
train_keys = set(train_data["key"]); assert not train_keys & {s["key"] for s in test_subset}, "test receipt found in training data"

def target_json(sample):
    return json.dumps({k: sample["entities"][k] for k in FIELDS}, ensure_ascii=False)

def to_conversation(sample):
    return {"messages": [
        {"role": "user", "content": [{"type": "image", "image": sample["image"].convert("RGB")},   # IMAGE FIRST
                                     {"type": "text", "text": INSTRUCTION}]},
        {"role": "assistant", "content": [{"type": "text", "text": target_json(sample)}]},
    ]}

train_conversations = [to_conversation(s) for s in train_data]
print("example target:", target_json(train_data[0]))

## 4. Generation helper with the visual budget  (GATE B)

In [ ]:
import time
_budget_mode = None
def _encode(image, instruction):
    """Builds model inputs with the image first and tries to apply MAX_SOFT_TOKENS."""
    global _budget_mode
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": instruction}]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    img = image.convert("RGB")
    tries = [("images_kwargs", {"images_kwargs": {"max_soft_tokens": MAX_SOFT_TOKENS}}),
             ("max_soft_tokens", {"max_soft_tokens": MAX_SOFT_TOKENS}),
             ("default", {})]
    if _budget_mode is not None:
        tries = [t for t in tries if t[0] == _budget_mode]
    last_err = None
    for name, kw in tries:
        try:
            inputs = processor(img, prompt, add_special_tokens=False, return_tensors="pt", **kw)
            _budget_mode = name
            return inputs
        except (TypeError, ValueError) as e:
            last_err = e
    raise last_err

def n_image_tokens(inputs):
    return None if IMAGE_TOKEN_ID is None else int((inputs["input_ids"] == IMAGE_TOKEN_ID).sum())

@torch.inference_mode()
def generate_from_image(image, instruction=INSTRUCTION, max_new_tokens=MAX_NEW_TOKENS):
    inputs = _encode(image, instruction).to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True, do_sample=False)   # greedy, batch 1
    return processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

FastVisionModel.for_inference(model)
probe = _encode(test_subset[0]["image"], INSTRUCTION)
k = n_image_tokens(probe)
print("budget mode:", _budget_mode, "| image tokens:", k, "| prompt length:", probe["input_ids"].shape[1])
if k is not None and k <= 280:
    print("WARNING: <=280 image tokens, so the 560 budget was NOT applied. Fix before evaluating (see the runbook).")
t = time.time(); print("output:", generate_from_image(test_subset[0]["image"])); print(f"{time.time() - t:.1f} s per receipt")
print("expected:", target_json(test_subset[0]))
# GATE B: image tokens > 280 (budget applied), output is JSON-like, s/receipt known. Plan the evaluation time from s/receipt.

## 4b. Training-collator budget check  (GATE B2)

In [ ]:
# GATE B2: does the TRAINING collator apply the same visual budget? Settle one budget for everything BEFORE the baselines.
try:
    from unsloth.trainer import UnslothVisionDataCollator
    _c = UnslothVisionDataCollator(model, processor)
    _b = _c([train_conversations[0]])
    k_train = int((_b["input_ids"] == IMAGE_TOKEN_ID).sum()) if IMAGE_TOKEN_ID is not None else None
    k_infer = n_image_tokens(_encode(train_data[0]["image"], INSTRUCTION))
    print("image tokens, training collator:", k_train, "| inference:", k_infer)
    if None not in (k_train, k_infer) and k_train != k_infer:
        print("MISMATCH: training and inference see different budgets. Make them equal before the baselines (runbook, GATE B2).")
    del _c, _b
except Exception as e:
    print("collator check could not run before LoRA:", repr(e)[:300], "(it runs again in section 8)")

## 5. Scorer: strict, normalized, receipt-level

In [ ]:
import re, difflib
import pandas as pd
from dateutil import parser as dparser

def parse_json(text):
    text = re.sub(r"```(?:json)?", "", str(text))
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        return obj if isinstance(obj, dict) else None
    except json.JSONDecodeError:
        return None

def norm_text(s):
    return re.sub(r"\s+", " ", str(s or "")).strip().upper().strip(" .,;:")

def norm_total(s):
    nums = re.findall(r"\d+(?:\.\d+)?", str(s or "").replace(",", ""))
    return f"{float(nums[-1]):.2f}" if nums else ""

def norm_date(s):
    s = str(s or "").strip()
    if not s:
        return ""
    try:
        return dparser.parse(s, dayfirst=True, fuzzy=True).strftime("%Y-%m-%d")
    except Exception:
        return re.sub(r"\D", "", s)

NORMALIZERS = {"company": norm_text, "address": norm_text, "date": norm_date, "total": norm_total}
METRICS = ["valid_json"] + [f"{k}_{m}" for k in FIELDS for m in ("strict", "norm")] + ["all4_strict", "all4_norm"]

def score(prediction, gold):
    pred = parse_json(prediction)
    row = {"valid_json": int(pred is not None)}
    pred = pred or {}
    for k in FIELDS:
        p, g = str(pred.get(k, "") or "").strip(), str(gold[k] or "").strip()
        row[f"{k}_strict"] = int(p == g)
        row[f"{k}_norm"] = int(NORMALIZERS[k](g) != "" and NORMALIZERS[k](p) == NORMALIZERS[k](g))
        if k in ("company", "address"):
            row[f"{k}_sim"] = round(difflib.SequenceMatcher(None, norm_text(p), norm_text(g)).ratio(), 3)
    row["all4_strict"] = int(all(row[f"{k}_strict"] for k in FIELDS))
    row["all4_norm"] = int(all(row[f"{k}_norm"] for k in FIELDS))
    return row

def evaluate(samples, label, instruction=INSTRUCTION, plain=False):
    FastVisionModel.for_inference(model)
    rows, t0 = [], time.time()
    for i, s in enumerate(samples, 1):
        if plain and hasattr(model, "disable_adapter"):
            with model.disable_adapter():
                pred = generate_from_image(s["image"], instruction)
        else:
            pred = generate_from_image(s["image"], instruction)
        gold = s["entities"]
        rows.append({"key": s.get("key", i), "prediction": pred,
                     "expected": json.dumps({k: gold[k] for k in FIELDS}, ensure_ascii=False), **score(pred, gold)})
        if i % 10 == 0 or i == len(samples):
            print(f"\r{label}: {i}/{len(samples)} ({(time.time() - t0) / i:.1f} s/receipt)", end="")
    df = pd.DataFrame(rows)
    df.to_csv(f"{OUT}/eval_{label}.csv", index=False)
    print("\n" + df[METRICS].mean().mul(100).round(1).to_string())
    return df

## 6. Baseline 1: plain model, short instruction

In [ ]:
plain_df = evaluate(test_subset, "plain")

## 7. Baseline 2: plain model, detailed prompt (bonus: what fine-tuning adds beyond prompting)

In [ ]:
prompted_df = evaluate(test_subset, "prompted", instruction=PROMPT_DETAILED)

## 8. Add LoRA adapters and check one collated batch  (GATE C)

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0, bias="none", random_state=3407,
)
from unsloth.trainer import UnslothVisionDataCollator
collator = UnslothVisionDataCollator(model, processor)
batch = collator([train_conversations[0]])
labels = batch["labels"][0]
kept = labels[labels != -100]
print("sequence length:", batch["input_ids"].shape[1],
      "| image tokens:", int((batch["input_ids"] == IMAGE_TOKEN_ID).sum()) if IMAGE_TOKEN_ID is not None else "n/a",
      "| trained tokens:", kept.numel())
print("trained text:", processor.tokenizer.decode(kept[:300]))
# GATE C: image tokens > 280 here too (else the collator ignores the budget: record it, or set the budget the way
# that worked in section 4); the trained text should contain the JSON answer.

## 9. Smoke training: 5 steps  (GATE D)

In [ ]:
from trl import SFTTrainer, SFTConfig
def make_trainer(max_steps, out_dir, save=False):
    return SFTTrainer(
        model=model, train_dataset=train_conversations, processing_class=processor.tokenizer,
        data_collator=collator,
        args=SFTConfig(
            per_device_train_batch_size=1, gradient_accumulation_steps=4, max_grad_norm=0.3,
            warmup_ratio=0.03, max_steps=max_steps, learning_rate=LEARNING_RATE, logging_steps=1 if max_steps <= 10 else 5,
            optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="cosine", seed=3407,
            output_dir=out_dir, report_to="none",
            save_strategy="steps" if save else "no", save_steps=SAVE_STEPS,
            remove_unused_columns=False, dataset_text_field="",                 # required for vision fine-tuning
            dataset_kwargs={"skip_prepare_dataset": True}, max_length=2048,
        ),
    )
FastVisionModel.for_training(model)
smoke = make_trainer(SMOKE_STEPS, f"{OUT}/smoke").train()
sps = smoke.metrics["train_runtime"] / SMOKE_STEPS
print(f"{sps:.1f} s/step -> full run of {TRAIN_STEPS} steps ~ {sps * TRAIN_STEPS / 60:.0f} min | peak GPU {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
# GATE D: losses are finite (a first loss of 13-15 is NORMAL for Gemma 4). If the full run will not fit the
# time left, lower TRAIN_STEPS (and record the change).

## 10. Full training run

In [ ]:
FastVisionModel.for_training(model)
stats = make_trainer(TRAIN_STEPS, f"{OUT}/outputs", save=True).train()
print(f"Training finished in {stats.metrics['train_runtime'] / 60:.1f} min | peak GPU {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
print(f"Note for the settings table: {SMOKE_STEPS} smoke steps + {TRAIN_STEPS} steps.")

## 11. Save the adapters right away (never merged weights)

In [ ]:
ADAPTER_DIR = f"{OUT}/gemma4_sroie_lora"
model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)
!cd {OUT} && zip -qr gemma4_sroie_lora.zip gemma4_sroie_lora
print("Saved", ADAPTER_DIR, "and a zip. Download it or push it to the Hub now.")
HF_REPO = ""   # optional, e.g. "your-username/gemma4-e2b-sroie-lora" (needs HF_TOKEN with write access in Colab Secrets)
if HF_REPO:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    model.push_to_hub(HF_REPO, token=tok); processor.push_to_hub(HF_REPO, token=tok)
    print("Pushed to https://huggingface.co/" + HF_REPO)

## 12. Fine-tuned evaluation and comparison (same 100 receipts, same scorer)

In [ ]:
ft_df = evaluate(test_subset, "finetuned")
comparison = pd.DataFrame({
    "Plain (%)": plain_df[METRICS].mean() * 100,
    "Prompted (%)": prompted_df[METRICS].mean() * 100,
    "Fine-tuned (%)": ft_df[METRICS].mean() * 100,
}).round(1)
comparison.to_csv(f"{OUT}/comparison.csv")
print(comparison.to_string())

from scipy.stats import binomtest
def mcnemar(base, new):
    better = int(((base == 0) & (new == 1)).sum()); worse = int(((base == 1) & (new == 0)).sum())
    p = binomtest(better, better + worse, 0.5).pvalue if better + worse else 1.0
    return better, worse, p
for name, base in (("plain", plain_df), ("prompted", prompted_df)):
    for m in ("all4_norm", "all4_strict"):
        b, w, p = mcnemar(base[m].values, ft_df[m].values)
        print(f"fine-tuned vs {name} on {m}: {b} receipts improved, {w} got worse, exact McNemar p = {p:.2g}")

import matplotlib.pyplot as plt
show = [f"{k}_norm" for k in FIELDS] + ["all4_norm", "all4_strict"]
ax = comparison.loc[show].plot(kind="bar", figsize=(9, 4), rot=0)
ax.set_ylabel("% of 100 test receipts"); ax.set_title("Plain vs prompted vs fine-tuned Gemma 4 E2B on SROIE")
plt.tight_layout(); plt.savefig(f"{OUT}/comparison.png", dpi=150); plt.show()

## 13. Error analysis of the fine-tuned model (bonus)

In [ ]:
def error_type(row, k):
    if row[f"{k}_strict"]: return "correct"
    if row[f"{k}_norm"]: return "format only"
    pred = parse_json(row["prediction"]) or {}
    return "missing" if not str(pred.get(k, "")).strip() else "wrong value"
err = pd.DataFrame({k: ft_df.apply(lambda r: error_type(r, k), axis=1) for k in FIELDS})
print(err.apply(pd.Series.value_counts).fillna(0).astype(int).to_string())
for k in FIELDS:
    bad = ft_df[err[k] == "wrong value"].head(3)
    for _, r in bad.iterrows():
        print(f"\n[{k}] expected: {json.loads(r['expected'])[k]!r}\n      predicted: {(parse_json(r['prediction']) or {}).get(k, '')!r}")

## 14. Robustness check on local receipts (bonus: overfitting check)
Upload photos to `/content/hackday/local/`, plus `labels.csv` with the columns `file,company,date,address,total`, labelled by hand. **Never train on these.**

In [ ]:
import glob
from PIL import Image
lab_path = f"{OUT}/local/labels.csv"
if os.path.exists(lab_path):
    lab = pd.read_csv(lab_path).fillna("")
    local = [{"key": r["file"], "image": Image.open(f"{OUT}/local/{r['file']}"),
              "entities": {k: str(r[k]) for k in FIELDS}} for _, r in lab.iterrows()]
    local_plain = evaluate(local, "local_plain", plain=True)
    local_ft = evaluate(local, "local_finetuned")
    print(pd.DataFrame({"Plain (%)": local_plain[METRICS].mean() * 100, "Fine-tuned (%)": local_ft[METRICS].mean() * 100}).round(1).to_string())
else:
    print("No local receipts yet:", lab_path)

## 15. The app: receipt to expense, fine-tuned beside plain, with a public link

In [ ]:
import gradio as gr
FastVisionModel.for_inference(model)
os.makedirs(f"{OUT}/examples", exist_ok=True)
example_paths = []
for i, s in enumerate(test_subset[:3]):
    p = f"{OUT}/examples/receipt_{i}.jpg"; s["image"].convert("RGB").save(p); example_paths.append([p])
expenses = []

def extract(image):
    if image is None:
        raise gr.Error("Upload or photograph a receipt first.")
    t = time.time(); ft_raw = generate_from_image(image); t_ft = time.time() - t
    with model.disable_adapter():
        plain_raw = generate_from_image(image)
    f = parse_json(ft_raw) or {}
    flags = [n for n, ok in (("total", bool(norm_total(f.get("total", "")))), ("date", bool(re.search(r"\d", str(f.get("date", "")))))) if not ok]
    status = f"Fine-tuned answer in {t_ft:.1f} s" + (f"  |  please check: {', '.join(flags)}" if flags else "")
    return f.get("company", ""), f.get("date", ""), f.get("address", ""), f.get("total", ""), ft_raw, plain_raw, status

def add_expense(company, date, address, total):
    expenses.append({"company": company, "date": date, "address": address, "total": total})
    df = pd.DataFrame(expenses)
    spend = sum(float(norm_total(x) or 0) for x in df["total"])
    path = f"{OUT}/expenses.csv"; df.to_csv(path, index=False)
    return df, f"**Total spend: {spend:.2f}**", path

gr.close_all()
with gr.Blocks(title="Receipt to Expense") as demo:
    gr.Markdown("# Receipt to Expense\nGemma 4 E2B fine-tuned with LoRA on SROIE receipts, shown beside the plain model.")
    with gr.Row():
        img = gr.Image(type="pil", sources=["upload", "webcam", "clipboard"], label="Receipt")
        with gr.Column():
            company = gr.Textbox(label="Company"); date = gr.Textbox(label="Date")
            address = gr.Textbox(label="Address", lines=2); total = gr.Textbox(label="Total")
            status = gr.Markdown()
    btn = gr.Button("Extract", variant="primary")
    with gr.Row():
        ft_box = gr.Textbox(label="Fine-tuned Gemma (raw output)", lines=5)
        plain_box = gr.Textbox(label="Plain Gemma (raw output)", lines=5)
    add_btn = gr.Button("Add to expenses")
    table = gr.Dataframe(label="Expenses"); spend = gr.Markdown(); csv_out = gr.File(label="Download CSV")
    gr.Examples(example_paths, inputs=img)
    btn.click(extract, inputs=img, outputs=[company, date, address, total, ft_box, plain_box, status])
    add_btn.click(add_expense, inputs=[company, date, address, total], outputs=[table, spend, csv_out])
demo.launch(share=True)

## 16. Optional: model-size comparison with E4B (bonus)
In a **second runtime** (G4 is ideal), set `MODEL_NAME = "unsloth/gemma-4-E4B-it"` in section 1 and run sections 0-12. Report the following for E2B and E4B:
- all-four accuracy;
- seconds per receipt;
- peak GPU memory.

## 17. Summary for the slides

In [ ]:
summary = {
    "model": MODEL_NAME, "load_in_4bit": LOAD_IN_4BIT, "visual_budget": MAX_SOFT_TOKENS, "budget_mode": _budget_mode,
    "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "steps": f"{SMOKE_STEPS} smoke + {TRAIN_STEPS}", "lr": LEARNING_RATE,
    "batch": "1 x grad-accum 4", "train_receipts": len(train_data), "test_receipts": N_TEST, "seed": SEED,
    "gpu": torch.cuda.get_device_name(0),
}
print(json.dumps(summary, indent=1))
print(comparison.to_string())